# Interior Product Binder Builder

A point-and-click front-end for the binder generator — no command line needed.

**How to use:**
1. Run the **Setup** cell below once (*Run → Run All*, or Shift+Enter).
2. Pick a mode:
   - **Manual entry** — type each product (KEY, Description, Source), click
     **Add row**. Use each row's **Edit** / **Delete** buttons to fix mistakes.
   - **Import Excel** — browse to an existing schedule `.xlsx` that already has
     all the information populated.
3. Fill in the **Project details**, tick or untick **Draft binder**, and click
   **Generate binder**. Rows that share the same Source are grouped onto one
   page; sources that are PDFs are embedded unchanged with the key notes and
   footer added on top.

> Requires `ANTHROPIC_API_KEY` in a `.env` file (see the README) for webpage
> sources; binders built purely from PDF sources work without it.


In [ ]:
# --- Setup: run this cell once ---
import os

# Make sure we're at the repo root so `src`, `output/`, and `.env` resolve.
if not os.path.exists("main.py") and os.path.exists(os.path.join("..", "main.py")):
    os.chdir("..")

from dotenv import load_dotenv
load_dotenv()

from src import excel_reader, pipeline

print("Working directory:", os.getcwd())
print("ANTHROPIC_API_KEY set:", bool(os.environ.get("ANTHROPIC_API_KEY")))
if not os.environ.get("ANTHROPIC_API_KEY"):
    print("  -> Needed only for webpage sources; PDF-only binders work without it.")


In [ ]:
# --- Binder form ---
import os
import ipywidgets as widgets
from IPython.display import display, clear_output

rows = []            # manual-mode product rows
state = {"editing": None}   # index of the row being edited, or None
FULL = widgets.Layout(width="90%")

# --- Shared: project details + draft toggle ---
project_name = widgets.Text(value="Untitled Project", description="Project:",
                            layout=widgets.Layout(width="70%"))
prepared_by = widgets.Text(value=os.environ.get("COMPANY_NAME", ""),
                           description="Prepared by:",
                           layout=widgets.Layout(width="70%"))
draft_chk = widgets.Checkbox(
    value=True, description="Draft binder (DRAFT watermark + NOT FOR CONSTRUCTION)",
    indent=False, layout=widgets.Layout(width="90%"))

# --- Mode switch ---
mode = widgets.ToggleButtons(options=["Manual entry", "Import Excel"],
                             description="Mode:")

# --- Manual-entry widgets ---
key_in = widgets.Text(description="KEY", placeholder="e.g. U-36")
desc_in = widgets.Text(description="Description",
                       placeholder="e.g. VANITY CABINET - 60-IN", layout=FULL)
source_in = widgets.Text(description="Source",
                         placeholder="local PDF path or URL (blank = owner input needed)",
                         layout=widgets.Layout(width="72%"))
browse_btn = widgets.Button(description="Browse...", icon="folder-open",
                            layout=widgets.Layout(width="16%"))
pg_in = widgets.Text(description="Page (PG)", placeholder="optional; same value groups sourceless rows onto one page")
desc2_in = widgets.Text(description="Description 2", placeholder="optional", layout=FULL)
desc3_in = widgets.Text(description="Description 3", placeholder="optional", layout=FULL)
optional_acc = widgets.Accordion(children=[widgets.VBox([pg_in, desc2_in, desc3_in])])
optional_acc.set_title(0, "Optional details (PG, Description 2, Description 3)")
optional_acc.selected_index = None

add_btn = widgets.Button(description="Add row", button_style="info", icon="plus")
cancel_btn = widgets.Button(description="Cancel edit",
                            layout=widgets.Layout(display="none"))
clear_btn = widgets.Button(description="Clear all rows", button_style="warning")
table_box = widgets.VBox([])

FIELDS = [key_in, desc_in, source_in, pg_in, desc2_in, desc3_in]
FIELD_KEYS = ["KEY", "DESCRIPTION", "PATH", "PG", "DESCRIPTION2", "DESCRIPTION3"]

# --- Import-Excel widgets ---
xlsx_in = widgets.Text(description="Schedule:",
                       placeholder="path to a populated .xlsx schedule",
                       layout=widgets.Layout(width="72%"))
xlsx_browse_btn = widgets.Button(description="Browse...", icon="folder-open",
                                 layout=widgets.Layout(width="16%"))

# --- Generate + status ---
gen_btn = widgets.Button(description="Generate binder", button_style="success",
                         icon="check")
status_out = widgets.Output()


def _pick_file(target, title, filetypes):
    try:
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        path = filedialog.askopenfilename(title=title, filetypes=filetypes)
        root.destroy()
        if path:
            target.value = path
    except Exception as e:
        with status_out:
            print(f"File picker unavailable ({e}). Please paste the path manually.")


def on_browse(_):
    _pick_file(source_in, "Select a product spec sheet (PDF)",
               [("PDF files", "*.pdf"), ("All files", "*.*")])


def on_xlsx_browse(_):
    _pick_file(xlsx_in, "Select a schedule spreadsheet",
               [("Excel files", "*.xlsx"), ("All files", "*.*")])


def _fill_fields(row):
    for w, k in zip(FIELDS, FIELD_KEYS):
        w.value = row.get(k, "")


def _clear_fields():
    for w in FIELDS:
        w.value = ""


def _set_editing(idx):
    state["editing"] = idx
    if idx is None:
        add_btn.description, add_btn.button_style = "Add row", "info"
        cancel_btn.layout.display = "none"
        _clear_fields()
    else:
        add_btn.description, add_btn.button_style = f"Update row {idx + 1}", "primary"
        cancel_btn.layout.display = ""
        _fill_fields(rows[idx])
    render_table()


def render_table():
    children = []
    for i, r in enumerate(rows):
        marker = " (editing)" if state["editing"] == i else ""
        src = r["PATH"] or "(owner input needed)"
        label = widgets.HTML(
            f"<b>{i + 1}. {r['KEY']}</b> — {r['DESCRIPTION'] or '<i>no description</i>'}"
            f" — <span style='color:#666'>{src}</span>{marker}")
        edit_b = widgets.Button(description="Edit", layout=widgets.Layout(width="60px"))
        del_b = widgets.Button(description="Delete", button_style="danger",
                               layout=widgets.Layout(width="70px"))

        def on_edit(_, idx=i):
            _set_editing(idx)

        def on_del(_, idx=i):
            rows.pop(idx)
            if state["editing"] is not None:
                _set_editing(None)
            else:
                render_table()

        edit_b.on_click(on_edit)
        del_b.on_click(on_del)
        children.append(widgets.HBox([edit_b, del_b, label]))
    if not children:
        children = [widgets.HTML("<i>No rows yet — add a product above.</i>")]
    table_box.children = tuple(children)


def on_add(_):
    with status_out:
        clear_output()
        if not key_in.value.strip():
            print("KEY is required.")
            return
    row = {k: w.value.strip() for w, k in zip(FIELDS, FIELD_KEYS)}
    if state["editing"] is None:
        rows.append(row)
        _clear_fields()
        render_table()
    else:
        rows[state["editing"]] = row
        _set_editing(None)


def on_cancel(_):
    _set_editing(None)


def on_clear(_):
    rows.clear()
    _set_editing(None)
    with status_out:
        clear_output()


def on_generate(_):
    with status_out:
        clear_output()
        os.makedirs("output", exist_ok=True)
        base = pipeline.unique_output_base(
            project_name.value or "Untitled Project", prepared_by.value)
        pdf_path = base + ".pdf"
        if mode.value == "Manual entry":
            if not rows:
                print("Add at least one product row first.")
                return
            xlsx_path = base + ".xlsx"
            excel_reader.save_schedule(rows, xlsx_path)
            print(f"Saved schedule: {xlsx_path}\n")
        else:
            xlsx_path = xlsx_in.value.strip()
            if not xlsx_path or not os.path.exists(xlsx_path):
                print("Choose an existing .xlsx schedule first (Browse...).")
                return
        try:
            results = pipeline.generate_from_xlsx(
                xlsx_path, pdf_path,
                project_name.value or "Untitled Project",
                prepared_by.value, log=print, draft=draft_chk.value)
        except PermissionError:
            print("Could not read the schedule - is it open in Excel? Close it and retry.")
            return
        ok = sum(1 for r in results if not r.error)
        print(f"\nDone: {ok}/{len(results)} product group(s) succeeded.")
        print(f"Binder PDF: {pdf_path}")


browse_btn.on_click(on_browse)
xlsx_browse_btn.on_click(on_xlsx_browse)
add_btn.on_click(on_add)
cancel_btn.on_click(on_cancel)
clear_btn.on_click(on_clear)
gen_btn.on_click(on_generate)

manual_panel = widgets.VBox([
    widgets.HBox([key_in, desc_in]),
    widgets.HBox([source_in, browse_btn]),
    optional_acc,
    widgets.HBox([add_btn, cancel_btn, clear_btn]),
    table_box,
])
load_rows_btn = widgets.Button(description="Load rows for review", icon="table")


def on_load_rows(_):
    with status_out:
        clear_output()
        path = xlsx_in.value.strip()
        if not path or not os.path.exists(path):
            print("Choose an existing .xlsx schedule first (Browse...).")
            return
        try:
            loaded = excel_reader.load_rows(path)
        except (ValueError, PermissionError) as e:
            print(f"Could not read schedule: {e}")
            return
        if not loaded:
            print("No rows with a KEY were found.")
            return
        rows.clear()
        rows.extend(loaded)
        _set_editing(None)
        mode.value = "Manual entry"
        print(f"Loaded {len(loaded)} row(s) for review - edit as needed, "
              "then Generate binder.")


load_rows_btn.on_click(on_load_rows)
import_panel = widgets.VBox([
    widgets.HBox([xlsx_in, xlsx_browse_btn]),
    load_rows_btn,
])
mode_panel = widgets.VBox([manual_panel])


def on_mode(change):
    mode_panel.children = (manual_panel,) if change["new"] == "Manual entry" else (import_panel,)


mode.observe(on_mode, names="value")

display(widgets.HTML("<h3>1. Project details</h3>"))
display(project_name, prepared_by, draft_chk)
display(widgets.HTML("<h3>2. Products</h3>"))
display(mode)
display(mode_panel)
display(widgets.HTML("<h3>3. Generate</h3>"))
display(gen_btn)
display(status_out)
render_table()
